# Data Audit

Prepare the repo inside Kaggle and inspect the locally generated Gemini-backed prompt inventory and split manifests. Data generation and augmentation are expected to happen before the repo is cloned into Kaggle.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("FRS_REPO_URL", "").strip()
REPO_DIR = Path("/kaggle/working/false-refusal-suppression")

if not REPO_URL:
    raise RuntimeError("Set FRS_REPO_URL in the Kaggle environment before running this notebook.")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(REPO_DIR)

In [ ]:
PILOT_PROMPTS = Path("data/processed/prompts/pilot_prompts_gemini.jsonl")
split_dir = Path("data/processed/splits/pilot_gemini")

required_paths = [
    PILOT_PROMPTS,
    split_dir / "discovery.jsonl",
    split_dir / "selection.jsonl",
    split_dir / "holdout.jsonl",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing prebuilt local data artifacts. Generate them locally before running Kaggle: " + ", ".join(missing)
    )

print(PILOT_PROMPTS)
print(split_dir)

In [ ]:
import json
import pandas as pd

records = []
with open(PILOT_PROMPTS, "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
display(df.groupby(["group", "topic"]).size().reset_index(name="count"))
display(df.groupby("family_id").size().reset_index(name="examples_per_family"))
display(
    pd.DataFrame(
        {
            "split": ["discovery", "selection", "holdout"],
            "path": [
                str(split_dir / "discovery.jsonl"),
                str(split_dir / "selection.jsonl"),
                str(split_dir / "holdout.jsonl"),
            ],
        }
    )
)